# OceanWatch Analytics · Entrega 1

Lakehouse del tráfico marítimo con datos AIS de NOAA (1–7 junio 2023). Decisiones técnicas y detalles en el `README.md` del repositorio.

# 1. Ingesta

### 1.0 Configuración

In [0]:
import hashlib
import json
import os
import shutil
import time
import urllib.error
import urllib.request
import zipfile
from datetime import datetime, timezone

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

CATALOG = "ocean_watch"
SCHEMA = "raw"
VOLUME = "ais_raw"
TABLE = f"{CATALOG}.{SCHEMA}.ais"

YEAR, MONTH = 2023, 6
DAYS = list(range(1, 8))
BASE_URL = "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/{year}/{stem}.zip"

VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
CSV_DIR = f"{VOLUME_ROOT}/csv"
MANIFEST_DIR = f"{VOLUME_ROOT}/_manifest"
LOCAL_TMP = "/tmp/ais_download"

MAX_RETRIES = 5
BACKOFF_BASE = 10
TIMEOUT = 120
CHUNK = 8 * 1024 * 1024

EXPECTED_COLUMNS = [
    "MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG", "Heading",
    "VesselName", "IMO", "CallSign", "VesselType", "Status",
    "Length", "Width", "Draft", "Cargo", "TransceiverClass",
]

spark.conf.set("spark.sql.session.timeZone", "UTC")


def file_stem(day: int) -> str:
    return f"AIS_{YEAR}_{MONTH:02d}_{day:02d}"


def humano(n: float) -> str:
    for unidad in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:,.1f} {unidad}"
        n /= 1024
    return f"{n:,.1f} TB"

### 1.1 Catálogo, esquema y Volume

In [0]:
spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {CATALOG}
    COMMENT 'OceanWatch Analytics: lakehouse del trafico maritimo (MINE 4213, 2026-20)'
""")
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}
    COMMENT 'Capa raw: datos AIS tal como se publican en NOAA Marine Cadastre, sin limpieza'
""")
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}
    COMMENT 'Archivos CSV diarios de AIS descomprimidos y manifiestos de ingesta'
""")

os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(MANIFEST_DIR, exist_ok=True)

### 1.2 Descarga, verificación de integridad y descompresión en el Volume

In [0]:
class IntegrityError(Exception):
    pass


def verify_zip(zip_path: str, expected_member: str) -> zipfile.ZipInfo:
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        if names != [expected_member]:
            raise IntegrityError(f"Contenido inesperado en el zip: {names}")
        bad = zf.testzip()
        if bad is not None:
            raise IntegrityError(f"CRC invalido en {bad}")
        return zf.getinfo(expected_member)


def download_with_retry(url: str, dest: str, expected_member: str) -> dict:
    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(f"  Intento {attempt}/{MAX_RETRIES}: {url}")
            req = urllib.request.Request(url, headers={"User-Agent": "ocean-watch-analytics/1.0"})
            sha = hashlib.sha256()
            with urllib.request.urlopen(req, timeout=TIMEOUT) as resp, open(dest, "wb") as f:
                content_length = resp.headers.get("Content-Length")
                etag = resp.headers.get("ETag")
                while chunk := resp.read(CHUNK):
                    f.write(chunk)
                    sha.update(chunk)

            received = os.path.getsize(dest)
            if content_length is not None and received != int(content_length):
                raise IntegrityError(f"Recibidos {received} bytes, se esperaban {content_length}")
            info = verify_zip(dest, expected_member)

            print(f"  OK: {humano(received)}, CRC verificado")
            return {
                "url": url,
                "etag": etag,
                "zip_bytes": received,
                "zip_sha256": sha.hexdigest(),
                "csv_bytes": info.file_size,
                "csv_crc32": f"{info.CRC:08x}",
                "attempts": attempt,
            }
        except urllib.error.HTTPError as exc:
            if 400 <= exc.code < 500 and exc.code != 429:
                raise
            last_exc = exc
        except (urllib.error.URLError, TimeoutError, OSError, IntegrityError, zipfile.BadZipFile) as exc:
            last_exc = exc

        print(f"  Fallo el intento {attempt}: {last_exc!r}")
        if attempt < MAX_RETRIES:
            wait = BACKOFF_BASE * 2 ** (attempt - 1)
            print(f"  Reintentando en {wait} s")
            time.sleep(wait)

    raise RuntimeError(f"No se pudo descargar {url} tras {MAX_RETRIES} intentos") from last_exc


def extract_to_volume(zip_path: str, member: str, dest_csv: str, expected_bytes: int) -> None:
    with zipfile.ZipFile(zip_path) as zf, zf.open(member) as src, open(dest_csv, "wb") as dst:
        shutil.copyfileobj(src, dst, CHUNK)
    written = os.path.getsize(dest_csv)
    if written != expected_bytes:
        raise IntegrityError(f"{dest_csv}: escritos {written} bytes, se esperaban {expected_bytes}")


def ingest_day(day: int) -> dict:
    stem = file_stem(day)
    url = BASE_URL.format(year=YEAR, stem=stem)
    csv_name = f"{stem}.csv"
    csv_path = f"{CSV_DIR}/{csv_name}"
    manifest_path = f"{MANIFEST_DIR}/{stem}.json"

    if os.path.exists(manifest_path) and os.path.exists(csv_path):
        with open(manifest_path) as f:
            manifest = json.load(f)
        if os.path.getsize(csv_path) == manifest["csv_bytes"]:
            print(f"  Ya ingerido y verificado, se omite ({humano(manifest['csv_bytes'])})")
            return {**manifest, "status": "skipped"}

    os.makedirs(LOCAL_TMP, exist_ok=True)
    zip_path = f"{LOCAL_TMP}/{stem}.zip"
    try:
        meta = download_with_retry(url, zip_path, csv_name)
        extract_to_volume(zip_path, csv_name, csv_path, meta["csv_bytes"])
    finally:
        if os.path.exists(zip_path):
            os.remove(zip_path)
    print(f"  Extraido en {csv_path} ({humano(meta['csv_bytes'])})")

    manifest = {
        "day": f"{YEAR}-{MONTH:02d}-{day:02d}",
        "file": csv_name,
        **meta,
        "ingested_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    return {**manifest, "status": "downloaded"}

In [0]:
results, failures = [], {}
for day in DAYS:
    print(f"\n--- {file_stem(day)} ---")
    try:
        results.append(ingest_day(day))
    except Exception as exc:
        failures[file_stem(day)] = repr(exc)
        print(f"  ERROR: {exc!r}")

if results:
    display(pd.DataFrame(results)[
        ["day", "status", "attempts", "zip_bytes", "csv_bytes", "csv_crc32", "zip_sha256", "ingested_at"]
    ])
if failures:
    raise RuntimeError(f"Fallaron {len(failures)} dias: {failures}")

total = 0
for fname in sorted(os.listdir(CSV_DIR)):
    size = os.path.getsize(f"{CSV_DIR}/{fname}")
    total += size
    print(f"{fname}  {humano(size)}")
print(f"Total en el Volume: {humano(total)}")

### 1.3 Validación del encabezado

In [0]:
for day in DAYS:
    path = f"{CSV_DIR}/{file_stem(day)}.csv"
    with open(path, encoding="utf-8") as f:
        header = f.readline().strip().lstrip("\ufeff").split(",")
    assert header == EXPECTED_COLUMNS, f"{path}: encabezado inesperado {header}"
print(f"Encabezado correcto en los {len(DAYS)} archivos")

### 1.4 Lectura con esquema explícito y carga en `ocean_watch.raw.ais`

In [0]:
ais_schema = StructType([
    StructField("MMSI",             StringType(),    nullable=True),
    StructField("BaseDateTime",     TimestampType(), nullable=True),
    StructField("LAT",              DoubleType(),    nullable=True),
    StructField("LON",              DoubleType(),    nullable=True),
    StructField("SOG",              DoubleType(),    nullable=True),
    StructField("COG",              DoubleType(),    nullable=True),
    StructField("Heading",          DoubleType(),    nullable=True),
    StructField("VesselName",       StringType(),    nullable=True),
    StructField("IMO",              StringType(),    nullable=True),
    StructField("CallSign",         StringType(),    nullable=True),
    StructField("VesselType",       IntegerType(),   nullable=True),
    StructField("Status",           IntegerType(),   nullable=True),
    StructField("Length",           DoubleType(),    nullable=True),
    StructField("Width",            DoubleType(),    nullable=True),
    StructField("Draft",            DoubleType(),    nullable=True),
    StructField("Cargo",            IntegerType(),   nullable=True),
    StructField("TransceiverClass", StringType(),    nullable=True),
    StructField("_corrupt_record",  StringType(),    nullable=True),
])
assert [f.name for f in ais_schema.fields[:-1]] == EXPECTED_COLUMNS

raw = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(ais_schema)
    .csv(f"{CSV_DIR}/*.csv")
)

ais = (
    raw
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("day", F.to_date("BaseDateTime"))
    .withColumn("ingested_at", F.current_timestamp())
)

(
    ais.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE)
)

### 1.5 Comentarios y metadatos de la tabla

In [0]:
COLUMN_COMMENTS = {
    "MMSI": "Maritime Mobile Service Identity. Identificador del buque, deberia tener 9 digitos",
    "BaseDateTime": "Fecha y hora UTC del reporte de posicion",
    "LAT": "Latitud en grados decimales, rango valido [-90, 90]",
    "LON": "Longitud en grados decimales, rango valido [-180, 180]",
    "SOG": "Speed over ground: velocidad sobre el fondo en nudos",
    "COG": "Course over ground: rumbo sobre el fondo en grados [0, 360)",
    "Heading": "Rumbo de la proa en grados [0, 359]. 511 significa no disponible",
    "VesselName": "Nombre del buque reportado por el transpondedor",
    "IMO": "Numero IMO con prefijo, por ejemplo IMO9074729. Vacio o IMO0000000 si no se reporta",
    "CallSign": "Indicativo de llamada de radio",
    "VesselType": "Codigo AIS de tipo de buque (ver catalogo de tipos de NOAA)",
    "Status": "Estado de navegacion AIS, codigos 0 a 15",
    "Length": "Eslora del buque en metros",
    "Width": "Manga del buque en metros",
    "Draft": "Calado del buque en metros",
    "Cargo": "Codigo AIS del tipo de carga",
    "TransceiverClass": "Clase del transpondedor AIS: A (buques comerciales) o B (embarcaciones menores)",
    "_corrupt_record": "Linea original del CSV cuando algun campo no se pudo convertir al tipo declarado; null si la fila se leyo bien",
    "source_file": "Archivo CSV de origen dentro del Volume",
    "day": "Fecha UTC derivada de BaseDateTime",
    "ingested_at": "Momento de la carga en la tabla",
}

spark.sql(f"""
    COMMENT ON TABLE {TABLE} IS
    'Posiciones AIS crudas de NOAA Marine Cadastre, 1 al 7 de junio de 2023. Una fila por mensaje de posicion, sin limpieza.'
""")
for column, comment in COLUMN_COMMENTS.items():
    spark.sql(f"ALTER TABLE {TABLE} ALTER COLUMN `{column}` COMMENT '{comment}'")

spark.sql(f"""
    ALTER TABLE {TABLE} SET TBLPROPERTIES (
        'source' = 'NOAA Marine Cadastre AIS',
        'source_url' = 'https://hub.marinecadastre.gov/pages/vesseltraffic',
        'coverage_start' = '{YEAR}-{MONTH:02d}-{DAYS[0]:02d}',
        'coverage_end' = '{YEAR}-{MONTH:02d}-{DAYS[-1]:02d}',
        'layer' = 'raw'
    )
""")

display(spark.sql(f"DESCRIBE TABLE EXTENDED {TABLE}"))

### 1.6 Reconciliación CSV vs tabla

In [0]:
csv_lines = (
    spark.read.text(f"{CSV_DIR}/*.csv")
    .where(F.length("value") > 0)
    .groupBy(F.col("_metadata.file_name").alias("source_file"))
    .agg((F.count("*") - 1).alias("csv_rows"))
)

table_rows = (
    spark.table(TABLE)
    .withColumn(
        "file_day",
        F.to_date(F.regexp_extract("source_file", r"(\d{4}_\d{2}_\d{2})", 1), "yyyy_MM_dd"),
    )
    .groupBy("source_file")
    .agg(
        F.count("*").alias("table_rows"),
        F.count("_corrupt_record").alias("corrupt_rows"),
        F.sum(F.when(F.col("day") != F.col("file_day"), 1).otherwise(0)).alias("rows_outside_file_day"),
        F.min("BaseDateTime").alias("min_ts"),
        F.max("BaseDateTime").alias("max_ts"),
    )
)

reconciliation = (
    csv_lines.join(table_rows, "source_file", "full")
    .withColumn("rows_match", F.coalesce(F.col("csv_rows") == F.col("table_rows"), F.lit(False)))
    .orderBy("source_file")
    .toPandas()
)
display(reconciliation)

print(f"Filas totales en {TABLE}: {reconciliation['table_rows'].sum():,}")
print(f"Filas con _corrupt_record: {reconciliation['corrupt_rows'].sum():,}")
assert len(reconciliation) == len(DAYS), f"Se esperaban {len(DAYS)} archivos, hay {len(reconciliation)}"
assert reconciliation["rows_match"].all(), "Hay archivos cuyo numero de filas no coincide con la tabla"

In [0]:
display(
    spark.table(TABLE)
    .where("_corrupt_record IS NOT NULL")
    .select("source_file", "_corrupt_record")
    .limit(20)
)

### 1.7 Limpieza de artefactos de la versión anterior (opcional)

In [0]:
CLEANUP_LEGACY = False

legacy_table = "workspace.default.ais"
legacy_files = [f"{VOLUME_ROOT}/ais_2023_06_{day:02d}" for day in DAYS]

print(f"{legacy_table} existe: {spark.catalog.tableExists(legacy_table)}")
print(f"Archivos antiguos en el Volume: {[p for p in legacy_files if os.path.exists(p)]}")

if CLEANUP_LEGACY:
    spark.sql(f"DROP TABLE IF EXISTS {legacy_table}")
    for path in legacy_files:
        if os.path.isdir(path):
            shutil.rmtree(path)
        elif os.path.exists(path):
            os.remove(path)

# 2. Exploración y perfilamiento

**Pendiente.**
- Posiciones por día y buques únicos por día.
- Distribución por tipo de buque (`VesselType`) y por tamaño (`Length`, `Width`).
- Cuantificar problemas de calidad: coordenadas fuera de rango, `SOG` imposibles, `Heading = 511`, MMSI que no tienen 9 dígitos, IMO vacíos o `IMO0000000`, duplicados exactos y por (`MMSI`, `BaseDateTime`), filas con `_corrupt_record`, filas fuera del día del archivo.

# 3. Preguntas de negocio

**Pendiente.** Cada respuesta con su plan de ejecución (`explain`).

### 3a. Buques distintos por día (`countDistinct` vs `approx_count_distinct`)

### 3b. Top 10 tipos de buque por número de posiciones y velocidad media

### 3c. Top 10 buques por distancia recorrida en la semana (haversine)

### 3d. Top 10 celdas de tráfico (H3 res. 8 o rejilla lat/lon) y cruce con el World Port Index

### 3e. Proporción de buques que transmitieron los 7 días y ubicación de los "visitantes de un solo día"

# 4. Almacenamiento óptimo para un propósito

**Pendiente.**
- Declarar el propósito de consulta.
- Comparar CSV vs Parquet vs Delta, y particionamiento vs `CLUSTER BY`.
- Evidencia antes y después: bytes y archivos leídos (plan o `INPUT_FILE_NAME`), efecto de `OPTIMIZE`.

# 5. Gobernanza

**Pendiente.** Esquemas adicionales en `ocean_watch` para las tablas derivadas, con comentarios en tablas y columnas.